In [ ]:
from googleapiclient.discovery import build
from google.oauth2 import service_account
from openai import OpenAI
import time

# Configuration
CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"
SPREADSHEET_ID = "1k5MwqNbHKRi1yXFfy-ZBsqy1pqOkkyKxGN1JBydRneo"
SHEET_NAME = "Sheet2"
OPENAI_API_KEY = ""

# Column mapping for your data
column_indices = {
	"FIRST_NAME": 0,  	# Column A
	"COMPANY_NAME": 3,  # Column D
	"ABOUT_US": 12,  		# M
	"EBOOK": 13,     		# N
	"COURSES": 14,   		# O
	"RECENT_BLOG": 15, 	# P
	"TESTIMONIALS": 16, # Q
	"WEBINAR": 17,   		# R
	"SERVICES": 18,  		# S
	"PODCAST": 19,   		# T
	"SHOP": 20       		# U 
}

# Output column for the generated email
OUTPUT_COLUMN = "W"

In [2]:
BASE_PROMPT = """
Objective: Generate initial cold emails openings for outreach, following the specific template provided below. Only modify the sections within square brackets for personalisation; all other content should remain fixed.
 
Instructions:
  - Personalization Fields:
    - Replace [FIRST NAME] with the name of the person given in the prompt.
    - Replace [PERSONALISATION] with a short, specific comment as per the instruction given in the square bracket of [PERSONALISATION - instruction here].
    - The short specific comment as mentioned above should provide some sort of insight and it should go along and tie in to the next line in the email template so that the overall email can make sense. 
    - While personalizing: Write the personalization in 3rd Grade level. The sentence should not be too long and complex. Use shorter sentences and simpler words.
  - Fixed Content:
    - Do not change any other text in the template. All non-bracketed content should remain exactly as written, preserving the wording, tone, and format. Be very very strict on this, I don't want anything else apart from the bracketed  content to change. 
  - Tone and Language:
    - Keep the tone friendly and professional.
  - Don't send anything else except for the Email opening in the output
  - Judge if the given data point is useful for the same, if not send "NONSENSICAL DATA POINT" in the output – VERY VERY IMPORTANT

Here's the template I'm using:

Saw you are helping [their ICP, be very specific with it based on the data points] with [very specific problem they're helping their ICP with], {{First Name}} 


[Compliment the Insight on how we think it's helping their ICP overcome that problem. Share what's the end benefits on how it's helping their ICP.]

===================

Examples based on this Template:

EXAMPLE - 1 : 

Saw you are helping companies with unconscious bias training, Marguerite.

I think it's great how you help people see their own biases. That makes workshops better for everyone

EXAMPLE - 2 :

Saw you are helping organizations improve their processes and build high-performing teams, Kevin.

I think it's wonderful how you help teams work together better and reach their goals. That is super important! 

EXAMPLE - 3 :

Saw you are helping communities in Uganda with access to healthcare, education, justice, and environmental sustainability, Steven.

I think it's wonderful how you empower children and transform lives. That is super important! 

===================

•⁠  ⁠Do the personalization using the pool of knowledge from the data points I'm giving you
•⁠  ⁠Looking at the data pool you'd be able to tell what's the profile of the prospect we're trying to reach out to and what kind of customers do they serve and what pain points do our prospects help their customers overcome.

•⁠  ⁠An Example of an Email that is strictly following the rules:

"Saw you are helping marketers adapt to the changes in digital privacy and data collection, Rydal.

It’s cool that you are helping them get ready for the cookieless future. That is super important!"

I really like the fact that the above email is sticking to the template and using the personalization on the first line alone as mentioned in the instructions.
"""

In [3]:
def authenticate_google_sheets():
    """Authenticate and return Google Sheets service"""
    try:
        creds = service_account.Credentials.from_service_account_file(
            CREDENTIALS_FILE, 
            scopes=['https://www.googleapis.com/auth/spreadsheets']
        )
        service = build('sheets', 'v4', credentials=creds)
        return service
    except Exception as e:
        print(f"Error authenticating Google Sheets: {e}")
        return None

def initialize_openai():
    """Initialize OpenAI client"""
    try:
        client = OpenAI(api_key=OPENAI_API_KEY)
        return client
    except Exception as e:
        print(f"Error initializing OpenAI: {e}")
        return None

def collect_all_usable_datapoints(row):
    """Collect all usable datapoints from a row"""
    datapoints = {}
    
    for category, col_index in column_indices.items():
        # Skip non-datapoint columns like name and company
        if category in ["FIRST_NAME", "COMPANY_NAME"]:
            continue
            
        if col_index < len(row):
            content = str(row[col_index]).strip()
            
            # Check if content is usable (not empty, not "no content", etc.)
            if (content and 
                content.lower() != "no content" and 
                content.lower() != "no meaningful content" and
                content.strip() != ""):
                datapoints[category] = content
    
    return datapoints

def generate_email(first_name, company_name, datapoints, client):
    """Generate email using all available datapoints"""
    try:
        # Prepare datapoints for the prompt
        datapoints_text = ""
        for category, content in datapoints.items():
            datapoints_text += f"{category}: {content}\n"
        
        prompt = f"""
        {BASE_PROMPT}
        
        Person's first name: {first_name}
        Company name: {company_name}
        
        Available datapoints to personalize with:
        {datapoints_text}
        
        Generate a personalized email using ALL the datapoints provided.
        """
        
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
            temperature=0.7
        )
        
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Error generating email: {e}")
        return f"Error generating email: {str(e)}"

def process_row(row, row_index, client):
    """Process a single row and generate an email"""
    try:
        # Extract basic information
        first_name = str(row[column_indices["FIRST_NAME"]]).strip() if len(row) > column_indices["FIRST_NAME"] else "Friend"
        company_name = str(row[column_indices["COMPANY_NAME"]]).strip() if len(row) > column_indices["COMPANY_NAME"] else "Your Company"
        
        print(f"Processing row {row_index + 2}: {first_name} from {company_name}")  # +2 to account for 0-indexing and header
        
        # Collect all usable datapoints
        datapoints = collect_all_usable_datapoints(row)
        print(f"  Found {len(datapoints)} usable datapoints")
        
        # Generate email with all datapoints
        email_content = generate_email(
            first_name,
            company_name,
            datapoints,
            client
        )
        
        return email_content
    
    except Exception as e:
        print(f"Error processing row {row_index + 2}: {e}")
        return f"Error: {str(e)}"

def update_sheet_with_results(service, results):
    """Update the Google Sheet with generated emails"""
    try:
        # Prepare batch update data
        data = []
        
        for row_index, email_content in results:
            if email_content is None:
                continue
                
            actual_row = row_index + 2  # Assuming header is row 1, data starts from row 2
            cell_range = f"{SHEET_NAME}!{OUTPUT_COLUMN}{actual_row}"
            
            data.append({
                'range': cell_range,
                'values': [[email_content]]
            })
        
        if data:
            body = {
                'valueInputOption': 'RAW',
                'data': data
            }
            
            result = service.spreadsheets().values().batchUpdate(
                spreadsheetId=SPREADSHEET_ID,
                body=body
            ).execute()
            
            print(f"Updated {len(data)} cells in the spreadsheet")
            return True
        else:
            print("No data to update")
            return False
    
    except Exception as e:
        print(f"Error updating sheet: {e}")
        return False

In [4]:
def main(start_row=1):
    """Main function to run the email generation process with batch processing
    Args:
        start_row (int): Row number to start processing from (1-based, excluding header)
    """
    print(f"Starting email generation process from row {start_row}...")
    
    # Initialize services
    sheets_service = authenticate_google_sheets()
    if not sheets_service:
        print("Failed to authenticate Google Sheets")
        return
        
    openai_client = initialize_openai()
    if not openai_client:
        print("Failed to initialize OpenAI client")
        return
        
    try:
        # Read data from Google Sheets
        range_name = f"{SHEET_NAME}!A:Z"  # Adjust range as needed to include all your columns
        result = sheets_service.spreadsheets().values().get(
            spreadsheetId=SPREADSHEET_ID,
            range=range_name
        ).execute()
        
        values = result.get('values', [])
        if not values:
            print('No data found in the sheet.')
            return
            
        # Skip header row and validate start_row
        data_rows = values[1:]  # Skip header
        total_rows = len(data_rows)
        print(f"Found {total_rows} data rows in the sheet")
        
        # Validate start_row
        if start_row < 1:
            print("Error: start_row must be >= 1")
            return
        elif start_row > total_rows:
            print(f"Error: start_row ({start_row}) is greater than total rows ({total_rows})")
            return
            
        # Adjust for 0-based indexing (start_row is 1-based)
        start_index = start_row - 1
        remaining_rows = data_rows[start_index:]
        remaining_count = len(remaining_rows)
        print(f"Starting from row {start_row}, processing {remaining_count} remaining rows")
        
        batch_size = 10
        results = []
        
        # Process rows in batches
        for batch_start in range(0, remaining_count, batch_size):
            batch_end = min(batch_start + batch_size, remaining_count)
            batch_rows = remaining_rows[batch_start:batch_end]
            
            print(f"\nProcessing batch: rows {start_row + batch_start} to {start_row + batch_end - 1}")
            
            # Process each row in the current batch
            batch_results = []
            for i, row in enumerate(batch_rows):
                actual_row_index = start_index + batch_start + i  # Global row index
                email_content = process_row(row, actual_row_index, openai_client)
                batch_results.append((actual_row_index, email_content))
                
                # Add delay to avoid rate limiting
                time.sleep(1)
            
            # Add to overall results
            results.extend(batch_results)
            
            # Update the sheet with this batch's results
            if update_sheet_with_results(sheets_service, batch_results):
                print(f"✅ Batch completed successfully!")
            else:
                print(f"❌ Failed to update batch")
                print("Stopping process to prevent data loss...")
                break
        
        print(f"\n🎉 Email generation completed! Successfully processed {len(results)} rows.")
    
    except Exception as e:
        print(f"Error in main process: {e}")

In [5]:
await main(start_row=9)

Starting email generation process from row 9...
Found 1003 data rows in the sheet
Starting from row 9, processing 995 remaining rows

Processing batch: rows 9 to 18
Processing row 10: Kim from Silicon Valley Golf Performance Center
  Found 7 usable datapoints
Processing row 11: Kelly from Simply Sales
  Found 4 usable datapoints
Processing row 12: Dave from DEXIOS SERVICES
  Found 2 usable datapoints
Processing row 13: Michael from ktMINE
  Found 1 usable datapoints
Processing row 14: Kevin from Craft & Art Wine and Spirits
  Found 0 usable datapoints
Processing row 15: Robert from The Training Center for Sales and Business Development
  Found 4 usable datapoints
Processing row 16: Paul from Sansome Pacific Properties
  Found 6 usable datapoints
Processing row 17: Lynn from Project 180
  Found 1 usable datapoints
Processing row 18: Ken from LINK Public Affairs
  Found 0 usable datapoints
Processing row 19: Anning from 6blu
  Found 0 usable datapoints
Updated 10 cells in the spreadsheet

TypeError: object NoneType can't be used in 'await' expression